In [ ]:
YCB28_OBJECTS = {
    "box": [
        "003_cracker_box",
        "004_sugar_box",
        "008_pudding_box",
        "009_gelatin_box",
        "010_potted_meat_can",
        "026_sponge",
        "036_wood_block",
        "061_foam_brick",
        "077_rubiks_cube",
    ],
    "cylinder": [
        "001_chips_can",
        "002_master_chef_can",
        "005_tomato_soup_can",
        "007_tuna_fish_can",
        "019_pitcher_base",
        "025_mug",
        "040_large_marker",
        "065_a_cups",
    ],
    "sphere": [
        "012_strawberry",
        "014_lemon",
        "015_peach",
        "017_orange",
        "018_plum",
        "054_softball",
        "055_baseball",
        "056_tennis_ball",
        "057_racquetball",
        "058_golf_ball",
        "063_a_marbles",
    ],
}

In [ ]:
# SETUP FILDER & LIB
# Setup folders and libraries

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys
import json
import tarfile
import shutil
import urllib.request
import numpy as np
import pandas as pd
from tqdm import tqdm

# Install libraries for reading mesh files
!pip -q install trimesh open3d

import trimesh
import open3d as o3d

PROJECT_DIR = Path("/content/drive/MyDrive/PointNet_APS_Project_V2")

YCB_DIR = PROJECT_DIR / "data" / "ycb28"
YCB_RAW_DIR = YCB_DIR / "raw_downloads"
YCB_EXTRACTED_DIR = YCB_DIR / "extracted_models"
YCB_PC_DIR = YCB_DIR / "generated_pointclouds"
YCB_METADATA_DIR = PROJECT_DIR / "metadata"

for d in [YCB_DIR, YCB_RAW_DIR, YCB_EXTRACTED_DIR, YCB_PC_DIR, YCB_METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("YCB folder:", YCB_DIR)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.5/745.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 82.1 MB/s eta 0:00:00
Project folder: /content/drive/MyDrive/PointNet_APS_Project_V2
YCB folder: /content/drive/MyDrive/PointNet_APS_Project_V2/data/ycb28


In [ ]:
# Check selected YCB-28 objects

label_map = {
    "box": 0,
    "cylinder": 1,
    "sphere": 2
}

records = []

for class_name, objects in YCB28_OBJECTS.items():
    for obj in objects:
        records.append({
            "object_name": obj,
            "class_name": class_name,
            "label": label_map[class_name]
        })

ycb28_objects_df = pd.DataFrame(records)

print("Number of selected YCB objects:", len(ycb28_objects_df))
display(ycb28_objects_df)

print("\nObjects per class:")
display(ycb28_objects_df["class_name"].value_counts())

expected_pointclouds = {
    "box": 9 * 20,
    "cylinder": 8 * 20,
    "sphere": 11 * 20
}

print("\nExpected final YCB-28 point clouds:")
print(expected_pointclouds)
print("Total expected:", sum(expected_pointclouds.values()))

Number of selected YCB objects: 28


,object_name,class_name,label
0,003_cracker_box,box,0
1,004_sugar_box,box,0
2,008_pudding_box,box,0
3,009_gelatin_box,box,0
4,010_potted_meat_can,box,0
5,026_sponge,box,0
6,036_wood_block,box,0
7,061_foam_brick,box,0
8,077_rubiks_cube,box,0
9,001_chips_can,cylinder,1



Objects per class:


,count
class_name,
sphere,11
box,9
cylinder,8



Expected final YCB-28 point clouds:
{'box': 180, 'cylinder': 160, 'sphere': 220}
Total expected: 560


In [ ]:
# Download YCB object mesh files for YCB-28
# Official YCB download first, with controlled fallback for 001_chips_can only

import urllib.request
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Make sure folders exist
YCB_RAW_DIR.mkdir(parents=True, exist_ok=True)
YCB_EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
YCB_METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Official YCB URL sources
BASE_URLS = [
    "http://ycb-benchmarks.s3-website-us-east-1.amazonaws.com/data/google",
    "https://ycb-benchmarks.s3.amazonaws.com/data/google",
]

# Some YCB objects have slightly different official archive names
OBJECT_NAME_CANDIDATES = {
    "001_chips_can": [
        "001_chips_can",
        "001_chips_can_16k",
    ],
    "065_a_cups": [
        "065_a_cups",
        "065-a_cups",
        "065-a-cups",
    ],
    "063_a_marbles": [
        "063_a_marbles",
        "063-a_marbles",
        "063-a-marbles",
    ],
}

def download_url(url, out_path):
    try:
        urllib.request.urlretrieve(url, out_path)
        if out_path.exists() and out_path.stat().st_size > 0:
            return True, ""
        return False, "Downloaded file is empty"
    except Exception as e:
        return False, str(e)

download_records = []

for obj in tqdm(ycb28_objects_df["object_name"].tolist()):
    success = False

    candidates = OBJECT_NAME_CANDIDATES.get(obj, [obj])

    # Standard official download attempts
    for candidate in candidates:
        for base_url in BASE_URLS:
            tgz_name = f"{candidate}_google_16k.tgz"
            url = f"{base_url}/{tgz_name}"

            # Save using original object name, so later code stays simple
            out_path = YCB_RAW_DIR / f"{obj}_google_16k.tgz"

            if out_path.exists() and out_path.stat().st_size > 0:
                download_records.append({
                    "object_name": obj,
                    "matched_candidate": candidate,
                    "source_type": "official_ycb_download",
                    "url": url,
                    "saved_as": str(out_path),
                    "status": "already_exists",
                    "error": ""
                })
                success = True
                break

            ok, error = download_url(url, out_path)

            download_records.append({
                "object_name": obj,
                "matched_candidate": candidate,
                "source_type": "official_ycb_download",
                "url": url,
                "saved_as": str(out_path),
                "status": "downloaded" if ok else "failed",
                "error": error
            })

            if ok:
                success = True
                break

        if success:
            break

    # Controlled fallback only for 001_chips_can
    if not success and obj == "001_chips_can":
        print("Official download failed for 001_chips_can. Trying public YCB fixed-mesh fallback...")

        try:
            !pip -q install huggingface_hub

            from huggingface_hub import snapshot_download

            chips_fallback_dir = YCB_EXTRACTED_DIR / "001_chips_can_hf_fixed_mesh"
            chips_fallback_dir.mkdir(parents=True, exist_ok=True)

            snapshot_download(
                repo_id="ll4ma-lab/ycb-fixed-meshes",
                repo_type="dataset",
                allow_patterns=["001_chips_can/**"],
                local_dir=str(chips_fallback_dir)
            )

            mesh_files = []
            for ext in [".obj", ".ply", ".stl"]:
                mesh_files.extend(list(chips_fallback_dir.rglob(f"*{ext}")))

            if len(mesh_files) > 0:
                download_records.append({
                    "object_name": obj,
                    "matched_candidate": "001_chips_can",
                    "source_type": "hf_ycb_fixed_mesh_fallback",
                    "url": "ll4ma-lab/ycb-fixed-meshes",
                    "saved_as": str(chips_fallback_dir),
                    "status": "downloaded",
                    "error": ""
                })
                success = True
            else:
                download_records.append({
                    "object_name": obj,
                    "matched_candidate": "001_chips_can",
                    "source_type": "hf_ycb_fixed_mesh_fallback",
                    "url": "ll4ma-lab/ycb-fixed-meshes",
                    "saved_as": str(chips_fallback_dir),
                    "status": "failed",
                    "error": "No mesh files found after fallback download"
                })

        except Exception as e:
            download_records.append({
                "object_name": obj,
                "matched_candidate": "001_chips_can",
                "source_type": "hf_ycb_fixed_mesh_fallback",
                "url": "ll4ma-lab/ycb-fixed-meshes",
                "saved_as": "",
                "status": "failed",
                "error": str(e)
            })

download_df = pd.DataFrame(download_records)

download_log_path = YCB_METADATA_DIR / "ycb28_download_log_final.csv"
download_df.to_csv(download_log_path, index=False)

print("Download log saved to:")
print(download_log_path)

print("\nDownload status counts:")
display(download_df["status"].value_counts())

display(download_df)

  4%|▎         | 1/28 [00:00<00:05,  4.61it/s]

Official download failed for 001_chips_can. Trying public YCB fixed-mesh fallback...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

100%|██████████| 28/28 [00:09<00:00,  2.88it/s]


Download log saved to:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/ycb28_download_log_final.csv

Download status counts:


,count
status,
already_exists,27
failed,4
downloaded,1


,object_name,matched_candidate,source_type,url,saved_as,status,error
0,003_cracker_box,003_cracker_box,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
1,004_sugar_box,004_sugar_box,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
2,008_pudding_box,008_pudding_box,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
3,009_gelatin_box,009_gelatin_box,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
4,010_potted_meat_can,010_potted_meat_can,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
5,026_sponge,026_sponge,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
6,036_wood_block,036_wood_block,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
7,061_foam_brick,061_foam_brick,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
8,077_rubiks_cube,077_rubiks_cube,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,already_exists,
9,001_chips_can,001_chips_can,official_ycb_download,http://ycb-benchmarks.s3-website-us-east-1.ama...,/content/drive/MyDrive/PointNet_APS_Project_V2...,failed,HTTP Error 404: Not Found


In [ ]:
# Final availability check for all 28 YCB objects

availability_records = []

for obj in ycb28_objects_df["object_name"].tolist():

    official_archive = YCB_RAW_DIR / f"{obj}_google_16k.tgz"
    fallback_dir = YCB_EXTRACTED_DIR / "001_chips_can_hf_fixed_mesh"

    if official_archive.exists() and official_archive.stat().st_size > 0:
        available = True
        source_type = "official_ycb_download"
        available_path = str(official_archive)

    elif obj == "001_chips_can" and fallback_dir.exists():
        mesh_files = []
        for ext in [".obj", ".ply", ".stl"]:
            mesh_files.extend(list(fallback_dir.rglob(f"*{ext}")))

        available = len(mesh_files) > 0
        source_type = "hf_ycb_fixed_mesh_fallback" if available else "missing"
        available_path = str(fallback_dir) if available else ""

    else:
        available = False
        source_type = "missing"
        available_path = ""

    availability_records.append({
        "object_name": obj,
        "available": available,
        "source_type": source_type,
        "available_path": available_path
    })

availability_df = pd.DataFrame(availability_records)

print("Available objects:", availability_df["available"].sum(), "/ 28")
display(availability_df)

print("\nMissing objects:")
display(availability_df[availability_df["available"] == False])

availability_path = YCB_METADATA_DIR / "ycb28_download_availability_check.csv"
availability_df.to_csv(availability_path, index=False)

print("Saved availability check:")
print(availability_path)


Available objects: 28 / 28


,object_name,available,source_type,available_path
0,003_cracker_box,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
1,004_sugar_box,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
2,008_pudding_box,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
3,009_gelatin_box,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
4,010_potted_meat_can,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
5,026_sponge,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
6,036_wood_block,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
7,061_foam_brick,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
8,077_rubiks_cube,True,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...
9,001_chips_can,True,hf_ycb_fixed_mesh_fallback,/content/drive/MyDrive/PointNet_APS_Project_V2...



Missing objects:


,object_name,available,source_type,available_path


Saved availability check:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/ycb28_download_availability_check.csv


In [ ]:
# Extract official YCB archives and create YCB-28 mesh manifest

import tarfile
from pathlib import Path
import pandas as pd

# Load availability check
availability_path = YCB_METADATA_DIR / "ycb28_download_availability_check.csv"
availability_df = pd.read_csv(availability_path)

# ---------- Extract official archives ----------
extract_records = []

for _, row in availability_df.iterrows():
    obj = row["object_name"]
    source_type = row["source_type"]
    available_path = row["available_path"]

    if source_type == "hf_ycb_fixed_mesh_fallback":
        extract_records.append({
            "object_name": obj,
            "status": "fallback_already_available",
            "source_type": source_type,
            "extract_dir": available_path
        })
        continue

    obj_extract_dir = YCB_EXTRACTED_DIR / obj
    obj_extract_dir.mkdir(parents=True, exist_ok=True)

    tgz_path = Path(available_path)

    if not tgz_path.exists():
        extract_records.append({
            "object_name": obj,
            "status": "missing_archive",
            "source_type": source_type,
            "extract_dir": str(obj_extract_dir)
        })
        continue

    existing_files = list(obj_extract_dir.rglob("*"))

    if len(existing_files) > 0:
        status = "already_extracted"
    else:
        try:
            with tarfile.open(tgz_path, "r:gz") as tar:
                tar.extractall(path=obj_extract_dir)
            status = "extracted"
        except Exception as e:
            status = f"failed: {e}"

    extract_records.append({
        "object_name": obj,
        "status": status,
        "source_type": source_type,
        "archive": str(tgz_path),
        "extract_dir": str(obj_extract_dir)
    })

extract_df = pd.DataFrame(extract_records)

print("Extraction status:")
display(extract_df["status"].value_counts())
display(extract_df)

extract_log_path = YCB_METADATA_DIR / "ycb28_extract_log_final.csv"
extract_df.to_csv(extract_log_path, index=False)
print("Saved extract log:", extract_log_path)


# ---------- Create mesh manifest ----------
def choose_best_mesh(search_dir):
    search_dir = Path(search_dir)

    priority_names = [
        "textured.obj",
        "nontextured.obj",
        "merged_cloud.ply",
        "textured.ply",
        "nontextured.ply",
        "nontextured.stl",
    ]

    all_meshes = []
    for ext in [".obj", ".ply", ".stl"]:
        all_meshes.extend(list(search_dir.rglob(f"*{ext}")))

    if len(all_meshes) == 0:
        return None

    for name in priority_names:
        matches = [p for p in all_meshes if p.name == name]
        if len(matches) > 0:
            return matches[0]

    all_meshes = sorted(all_meshes, key=lambda p: p.stat().st_size, reverse=True)
    return all_meshes[0]


mesh_records = []

for _, row in ycb28_objects_df.iterrows():
    obj = row["object_name"]
    class_name = row["class_name"]
    label = int(row["label"])

    source_type = availability_df.loc[
        availability_df["object_name"] == obj, "source_type"
    ].values[0]

    available_path = availability_df.loc[
        availability_df["object_name"] == obj, "available_path"
    ].values[0]

    if source_type == "hf_ycb_fixed_mesh_fallback":
        search_dir = Path(available_path)
    else:
        search_dir = YCB_EXTRACTED_DIR / obj

    mesh_path = choose_best_mesh(search_dir)

    mesh_records.append({
        "object_name": obj,
        "class_name": class_name,
        "label": label,
        "mesh_path": str(mesh_path) if mesh_path is not None else "",
        "mesh_found": mesh_path is not None,
        "mesh_file": mesh_path.name if mesh_path is not None else "",
        "source_type": source_type
    })

mesh_manifest_df = pd.DataFrame(mesh_records)

print("Meshes found:", mesh_manifest_df["mesh_found"].sum(), "/ 28")

print("\nObjects per class:")
display(mesh_manifest_df["class_name"].value_counts())

print("\nMissing meshes:")
display(mesh_manifest_df[mesh_manifest_df["mesh_found"] == False])

display(mesh_manifest_df)

mesh_manifest_path = YCB_METADATA_DIR / "ycb28_mesh_manifest.csv"
mesh_manifest_df.to_csv(mesh_manifest_path, index=False)

print("Saved mesh manifest:")
print(mesh_manifest_path)

Extraction status:


,count
status,
already_extracted,27
fallback_already_available,1


,object_name,status,source_type,archive,extract_dir
0,003_cracker_box,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
1,004_sugar_box,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
2,008_pudding_box,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
3,009_gelatin_box,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
4,010_potted_meat_can,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
5,026_sponge,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
6,036_wood_block,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
7,061_foam_brick,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
8,077_rubiks_cube,already_extracted,official_ycb_download,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
9,001_chips_can,fallback_already_available,hf_ycb_fixed_mesh_fallback,NaN,/content/drive/MyDrive/PointNet_APS_Project_V2...


Saved extract log: /content/drive/MyDrive/PointNet_APS_Project_V2/metadata/ycb28_extract_log_final.csv
Meshes found: 28 / 28

Objects per class:


,count
class_name,
sphere,11
box,9
cylinder,8



Missing meshes:


,object_name,class_name,label,mesh_path,mesh_found,mesh_file,source_type


,object_name,class_name,label,mesh_path,mesh_found,mesh_file,source_type
0,003_cracker_box,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
1,004_sugar_box,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
2,008_pudding_box,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
3,009_gelatin_box,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
4,010_potted_meat_can,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
5,026_sponge,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
6,036_wood_block,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
7,061_foam_brick,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
8,077_rubiks_cube,box,0,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,official_ycb_download
9,001_chips_can,cylinder,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,True,textured.obj,hf_ycb_fixed_mesh_fallback


Saved mesh manifest:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/ycb28_mesh_manifest.csv


In [ ]:
# Generate YCB-28 point-cloud CSV files
# 28 objects × 20 samples per object = 560 point clouds

import numpy as np
import pandas as pd
import trimesh
from pathlib import Path
from tqdm import tqdm

N_POINTS = 1000
N_SAMPLES_PER_OBJECT = 20
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

mesh_manifest_path = YCB_METADATA_DIR / "ycb28_mesh_manifest.csv"
mesh_manifest_df = pd.read_csv(mesh_manifest_path)

YCB_PC_DIR = YCB_DIR / "generated_pointclouds_bbox_norm"
YCB_PC_DIR.mkdir(parents=True, exist_ok=True)

def load_mesh_any(mesh_path):
    loaded = trimesh.load(mesh_path, process=True)

    if isinstance(loaded, trimesh.Scene):
        geometries = [g for g in loaded.geometry.values()]
        mesh = trimesh.util.concatenate(geometries)
    else:
        mesh = loaded

    if not isinstance(mesh, trimesh.Trimesh):
        raise ValueError(f"Could not load as Trimesh: {mesh_path}")

    if mesh.faces is None or len(mesh.faces) == 0:
        raise ValueError(f"Mesh has no faces: {mesh_path}")

    return mesh

def random_rotation_matrix(rng):
    roll, pitch, yaw = rng.uniform(0, 2 * np.pi, size=3)

    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(roll), -np.sin(roll)],
        [0, np.sin(roll), np.cos(roll)]
    ])

    Ry = np.array([
        [np.cos(pitch), 0, np.sin(pitch)],
        [0, 1, 0],
        [-np.sin(pitch), 0, np.cos(pitch)]
    ])

    Rz = np.array([
        [np.cos(yaw), -np.sin(yaw), 0],
        [np.sin(yaw), np.cos(yaw), 0],
        [0, 0, 1]
    ])

    return Rz @ Ry @ Rx

def normalise_xyz_bbox_style(xyz):
    # Move minimum x/y/z to zero
    xyz_min = xyz.min(axis=0)
    xyz_shifted = xyz - xyz_min

    # Scale using bounding-box diagonal
    xyz_max = xyz_shifted.max(axis=0)
    scale = np.sqrt(np.sum(xyz_max ** 2))

    if scale > 0:
        xyz_scaled = xyz_shifted / scale
    else:
        xyz_scaled = xyz_shifted

    # Centre centroid at origin
    centroid = xyz_scaled.mean(axis=0)
    xyz_centered = xyz_scaled - centroid

    return xyz_centered

generated_records = []
failed_records = []

for _, row in tqdm(mesh_manifest_df.iterrows(), total=len(mesh_manifest_df)):
    object_name = row["object_name"]
    class_name = row["class_name"]
    label = int(row["label"])
    mesh_path = row["mesh_path"]
    source_type = row["source_type"]

    output_dir = YCB_PC_DIR / class_name / object_name
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        mesh = load_mesh_any(mesh_path)

        for sample_id in range(N_SAMPLES_PER_OBJECT):
            out_path = output_dir / f"{object_name}_sample_{sample_id:02d}.csv"

            if out_path.exists():
                generated_records.append({
                    "path": str(out_path),
                    "object_name": object_name,
                    "class_name": class_name,
                    "label": label,
                    "sample_id": sample_id,
                    "variant": "ycb28",
                    "role": "ycb28_test",
                    "source_mesh": mesh_path,
                    "source_type": source_type,
                    "n_points": N_POINTS
                })
                continue

            # Sample 1000 points from mesh surface
            points, face_indices = trimesh.sample.sample_surface(mesh, N_POINTS)

            # Use face normals for sampled points
            normals = mesh.face_normals[face_indices]

            # Random roll-pitch-yaw rotation
            R = random_rotation_matrix(rng)
            points = points @ R.T
            normals = normals @ R.T

            # Normalise points
            points = normalise_xyz_bbox_style(points)

            # Normalise normals
            normal_lengths = np.linalg.norm(normals, axis=1, keepdims=True)
            normal_lengths[normal_lengths == 0] = 1
            normals = normals / normal_lengths

            # Match APS CSV format: x,y,z,nx,ny,nz,dummy
            dummy = np.zeros((N_POINTS, 1), dtype=np.float32)
            data = np.hstack([points, normals, dummy]).astype(np.float32)

            np.savetxt(out_path, data, delimiter=",")

            generated_records.append({
                "path": str(out_path),
                "object_name": object_name,
                "class_name": class_name,
                "label": label,
                "sample_id": sample_id,
                "variant": "ycb28",
                "role": "ycb28_test",
                "source_mesh": mesh_path,
                "source_type": source_type,
                "n_points": N_POINTS
            })

    except Exception as e:
        failed_records.append({
            "object_name": object_name,
            "class_name": class_name,
            "mesh_path": mesh_path,
            "error": str(e)
        })

ycb28_pc_manifest_df = pd.DataFrame(generated_records)
failed_df = pd.DataFrame(failed_records)

ycb28_pc_manifest_path = YCB_METADATA_DIR / "ycb28_pointcloud_manifest_bbox_norm.csv"
ycb28_pc_manifest_df.to_csv(ycb28_pc_manifest_path, index=False)

failed_path = YCB_METADATA_DIR / "ycb28_pointcloud_generation_failures_bbox_norm.csv"
failed_df.to_csv(failed_path, index=False)

print("Generated point-cloud rows:", len(ycb28_pc_manifest_df))
print("Expected:", 28 * 20)

print("\nFailures:", len(failed_df))
display(failed_df)

print("\nPoint clouds per class:")
display(ycb28_pc_manifest_df["class_name"].value_counts())

print("\nPoint clouds per object:")
display(
    ycb28_pc_manifest_df
    .groupby(["class_name", "object_name"])
    .size()
    .reset_index(name="count")
)

print("\nSaved YCB-28 point-cloud manifest:")
print(ycb28_pc_manifest_path)

 43%|████▎     | 12/28 [00:21<00:26,  1.67s/it]/usr/local/lib/python3.12/dist-packages/trimesh/grouping.py:99: RuntimeWarning: invalid value encountered in cast
  stacked = np.column_stack(stacked).round().astype(np.int64)
100%|██████████| 28/28 [00:58<00:00,  2.09s/it]

Generated point-cloud rows: 560
Expected: 560

Failures: 0


""



Point clouds per class:


,count
class_name,
sphere,220
box,180
cylinder,160



Point clouds per object:


,class_name,object_name,count
0,box,003_cracker_box,20
1,box,004_sugar_box,20
2,box,008_pudding_box,20
3,box,009_gelatin_box,20
4,box,010_potted_meat_can,20
5,box,026_sponge,20
6,box,036_wood_block,20
7,box,061_foam_brick,20
8,box,077_rubiks_cube,20
9,cylinder,001_chips_can,20



Saved YCB-28 point-cloud manifest:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/ycb28_pointcloud_manifest_bbox_norm.csv


In [ ]:
# Quick check one generated YCB-28 CSV file

sample_path = Path(ycb28_pc_manifest_df.iloc[0]["path"])
sample_data = np.loadtxt(sample_path, delimiter=",")

print("Sample file:", sample_path)
print("Shape:", sample_data.shape)
print("First row:", sample_data[0])

print("\nXYZ min:", sample_data[:, :3].min(axis=0))
print("XYZ max:", sample_data[:, :3].max(axis=0))

normal_lengths = np.linalg.norm(sample_data[:, 3:6], axis=1)
print("\nNormal length mean:", normal_lengths.mean())
print("Normal length min:", normal_lengths.min())
print("Normal length max:", normal_lengths.max())

Sample file: /content/drive/MyDrive/PointNet_APS_Project_V2/data/ycb28/generated_pointclouds_bbox_norm/box/003_cracker_box/003_cracker_box_sample_00.csv
Shape: (1000, 7)
First row: [ 0.1315825   0.0307808   0.09927841  0.59306449 -0.70776463  0.38385373
  0.        ]

XYZ min: [-0.29674709 -0.3047092  -0.25614905]
XYZ max: [0.30104923 0.30203155 0.2677834 ]

Normal length mean: 1.0000000004881862
Normal length min: 0.9999999600706416
Normal length max: 1.0000000368181532


In [ ]:
# Save final Notebook 4 status

import json
from pathlib import Path

status = {
    "notebook": "04_prepare_ycb_dataset",
    "status": "complete",
    "dataset": "YCB-28",
    "total_objects": 28,
    "samples_per_object": 20,
    "total_point_clouds": int(len(ycb28_pc_manifest_df)),
    "points_per_sample": 1000,
    "features_per_point": 7,
    "class_counts": ycb28_pc_manifest_df["class_name"].value_counts().to_dict(),
    "manifest_path": str(ycb28_pc_manifest_path),
    "failure_count": int(len(failed_df))
}

status_path = YCB_METADATA_DIR / "notebook4_ycb28_preparation_status.json"

with open(status_path, "w") as f:
    json.dump(status, f, indent=4)

print("Notebook 4 status saved:")
print(status_path)

print(json.dumps(status, indent=4))

Notebook 4 status saved:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/notebook4_ycb28_preparation_status.json
{
    "notebook": "04_prepare_ycb_dataset",
    "status": "complete",
    "dataset": "YCB-28",
    "total_objects": 28,
    "samples_per_object": 20,
    "total_point_clouds": 560,
    "points_per_sample": 1000,
    "features_per_point": 7,
    "class_counts": {
        "sphere": 220,
        "box": 180,
        "cylinder": 160
    },
    "manifest_path": "/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/ycb28_pointcloud_manifest_bbox_norm.csv",
    "failure_count": 0
}
